In [6]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class BertrandPricingEnv(gym.Env):
    """
    Two-firm simultaneous price-setting environment.

    Demand for firm i (linear demand Q = a - bP, split on ties):
        Q_i = a - b*p_i        if p_i <  p_j   (firm i undercuts -> full demand)
        Q_i = 0.5*(a - b*p_i)  if p_i == p_j   (tie -> split demand)
        Q_i = 0                if p_i >  p_j   (firm i priced out)

    Profit for firm i:
        pi_i = (p_i - marginal_cost) * Q_i

    Action space (per firm): Discrete(n_price_levels)
        action index -> price via a linear grid from 0 to max_price
    Observation: last round's two prices, normalized to [0, 1]
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, a=100.0, b=1.0, marginal_cost=10.0,
                 n_price_levels=25, max_price=None, max_steps=20):
        super().__init__()

        # --- configurable market parameters ---
        self.a = a                          # demand intercept
        self.b = b                          # demand slope
        self.marginal_cost = marginal_cost  # c
        self.n_price_levels = n_price_levels
        # if no ceiling given, default to the monopoly-ish price a/b so the
        # grid covers the economically meaningful range
        self.max_price = max_price if max_price is not None else (a / b)
        self.max_steps = max_steps

        # price grid: index -> actual price
        # Built in two halves so that marginal_cost is ALWAYS an exact grid
        # point, regardless of n_price_levels. This prevents the unit-test
        # failure where price_index_for(c) snaps to a nearby point and
        # produces small negative profits instead of exactly zero.
        #   lower half: n_price_levels//2 points from 0 up to (not including) c
        #   upper half: remaining points from c up to max_price (c included)
        n_lower = n_price_levels // 2
        n_upper = n_price_levels - n_lower
        lower = np.linspace(0.0, marginal_cost, n_lower, endpoint=False)
        upper = np.linspace(marginal_cost, self.max_price, n_upper)
        self.price_grid = np.concatenate([lower, upper])
        # sanity: update n_price_levels to match actual grid length
        self.n_price_levels = len(self.price_grid)

        # --- Gymnasium spaces ---
        # two firms, each picks one discrete price level
        self.action_space = spaces.MultiDiscrete([n_price_levels, n_price_levels])

        # observation: last prices for both firms, normalized to [0,1]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(2,), dtype=np.float32
        )

        self._step_count = 0
        self._last_prices = np.zeros(2, dtype=np.float32)

    # ------------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._step_count = 0
        # start both firms at price 0 (arbitrary, agents will overwrite immediately)
        self._last_prices = np.zeros(2, dtype=np.float32)
        observation = self._get_obs()
        info = {"prices": self._last_prices.copy()}
        return observation, info

    # ------------------------------------------------------------------
    def step(self, actions):
        """
        actions: array-like of two ints, each in [0, n_price_levels-1]
        """
        actions = np.asarray(actions)
        p1 = self.price_grid[actions[0]]
        p2 = self.price_grid[actions[1]]
        prices = np.array([p1, p2], dtype=np.float32)

        q1, q2 = self._demand(p1, p2)
        profit1 = (p1 - self.marginal_cost) * q1
        profit2 = (p2 - self.marginal_cost) * q2

        self._last_prices = prices
        self._step_count += 1

        observation = self._get_obs()
        reward = np.array([profit1, profit2], dtype=np.float32)

        terminated = False  # no natural "win/lose" end state
        truncated = self._step_count >= self.max_steps

        info = {
            "prices": prices,
            "quantities": np.array([q1, q2], dtype=np.float32),
            "profits": reward,
        }
        return observation, reward, terminated, truncated, info

    # ------------------------------------------------------------------
    def _demand(self, p1, p2):
        """Piecewise Bertrand demand split. Returns (q1, q2)."""
        def q_if_alone(p):
            return max(self.a - self.b * p, 0.0)

        if p1 < p2:
            return q_if_alone(p1), 0.0
        elif p2 < p1:
            return 0.0, q_if_alone(p2)
        else:  # tie -> split
            q = q_if_alone(p1)
            return 0.5 * q, 0.5 * q

    # ------------------------------------------------------------------
    def _get_obs(self):
        # normalize prices into [0,1] using max_price
        return (self._last_prices / self.max_price).astype(np.float32)

    # ------------------------------------------------------------------
    def render(self):
        p1, p2 = self._last_prices
        print(f"step={self._step_count}  p1={p1:.2f}  p2={p2:.2f}")

    # ------------------------------------------------------------------
    def price_index_for(self, price):
        """Helper: find the closest action index for a target price.
        Useful for unit testing against the analytical Nash price."""
        return int(np.argmin(np.abs(self.price_grid - price)))


# ======================================================================
# UNIT TESTS — analytical Nash price check
# ======================================================================
def run_unit_tests():
    # n_price_levels=50 gives proper resolution for the tournament AND
    # marginal_cost=10 is guaranteed to be on the grid by the constructor's
    # two-half build logic — no need to use a coarse grid as a workaround.
    env = BertrandPricingEnv(a=100, b=1, marginal_cost=10,
                              n_price_levels=50, max_price=100)
    env.reset()

    c = env.marginal_cost
    c_idx = env.price_index_for(c)

    print("=== TEST 1: both firms price at marginal cost (Bertrand NE) ===")
    obs, reward, term, trunc, info = env.step([c_idx, c_idx])
    print(f"prices={info['prices']}  profits={info['profits']}")
    assert np.allclose(info["profits"], [0, 0], atol=1.0), \
        "FAIL: profits should be ~0 when both firms price at marginal cost"
    print("PASS: both profits ~0 at p1=p2=c\n")

    print("=== TEST 2: firm 1 undercuts slightly below firm 2 ===")
    above_c_idx = env.price_index_for(c + 20)  # firm 2 prices above cost
    env.reset()
    obs, reward, term, trunc, info = env.step([c_idx, above_c_idx])
    print(f"prices={info['prices']}  quantities={info['quantities']}  profits={info['profits']}")
    assert info["quantities"][1] == 0.0, \
        "FAIL: higher-priced firm should get zero demand"
    assert info["quantities"][0] > 0.0, \
        "FAIL: lower-priced firm should capture all demand"
    print("PASS: lower price captures full market, higher price gets zero\n")

    print("=== TEST 3: tie -> demand splits evenly ===")
    tie_idx = env.price_index_for(c + 20)
    env.reset()
    obs, reward, term, trunc, info = env.step([tie_idx, tie_idx])
    print(f"prices={info['prices']}  quantities={info['quantities']}")
    assert np.isclose(info["quantities"][0], info["quantities"][1]), \
        "FAIL: tied prices should split demand equally"
    print("PASS: tied prices split demand 50/50\n")

    print("=== TEST 4: episode truncates after max_steps ===")
    env2 = BertrandPricingEnv(max_steps=3)
    env2.reset()
    trunc = False
    steps_taken = 0
    while not trunc:
        _, _, _, trunc, _ = env2.step([5, 5])
        steps_taken += 1
    assert steps_taken == 3, f"FAIL: expected truncation at 3 steps, got {steps_taken}"
    print(f"PASS: episode truncated after {steps_taken} steps\n")

    print("ALL TESTS PASSED")


if __name__ == "__main__":
    run_unit_tests()

=== TEST 1: both firms price at marginal cost (Bertrand NE) ===
prices=[10. 10.]  profits=[0. 0.]
PASS: both profits ~0 at p1=p2=c

=== TEST 2: firm 1 undercuts slightly below firm 2 ===
prices=[10.   28.75]  quantities=[90.  0.]  profits=[0. 0.]
PASS: lower price captures full market, higher price gets zero

=== TEST 3: tie -> demand splits evenly ===
prices=[28.75 28.75]  quantities=[35.625 35.625]
PASS: tied prices split demand 50/50

=== TEST 4: episode truncates after max_steps ===
PASS: episode truncated after 3 steps

ALL TESTS PASSED


In [7]:
import csv
import numpy as np

# ======================================================================
# Agent base class
# ======================================================================
class BaseAgent:
    def __init__(self, name, price_grid):
        self.name = name
        self.price_grid = price_grid

    def reset(self):
        """Override if the agent needs to clear memory between episodes."""
        pass

    def act(self, history):
        """
        history: list of dicts, one per past round, each like
            {"my_price": float, "opp_price": float}
        (most recent round is history[-1])

        Returns: an integer action index into price_grid.
        """
        raise NotImplementedError

    def _index_for_price(self, price):
        return int(np.argmin(np.abs(self.price_grid - price)))


# ======================================================================
# The four agents
# ======================================================================
class AlwaysNashAgent(BaseAgent):
    """Always prices at marginal cost (the Bertrand NE)."""
    def __init__(self, name, price_grid, marginal_cost):
        super().__init__(name, price_grid)
        self._action = self._index_for_price(marginal_cost)

    def act(self, history):
        return self._action


class AlwaysColludeAgent(BaseAgent):
    """Always prices at the monopoly price, regardless of opponent."""
    def __init__(self, name, price_grid, monopoly_price):
        super().__init__(name, price_grid)
        self._action = self._index_for_price(monopoly_price)

    def act(self, history):
        return self._action


class TitForTatAgent(BaseAgent):
    """
    Round 1: price at the collusive (monopoly) level — be "nice" first.
    Every round after: mirror the opponent's previous price exactly.
    This implements all four Axelrod properties:
        nice (starts cooperative), retaliating (mirrors defection immediately),
        forgiving (mirrors cooperation immediately), clear (one-line rule).
    """
    def __init__(self, name, price_grid, monopoly_price):
        super().__init__(name, price_grid)
        self._cooperate_action = self._index_for_price(monopoly_price)

    def act(self, history):
        if len(history) == 0:
            return self._cooperate_action
        last_opp_price = history[-1]["opp_price"]
        return self._index_for_price(last_opp_price)


class RandomAgent(BaseAgent):
    """Uniformly random price each round."""
    def __init__(self, name, price_grid, seed):
        super().__init__(name, price_grid)
        self.rng = np.random.default_rng(seed)
        self._n_levels = len(price_grid)

    def act(self, history):
        return int(self.rng.integers(0, self._n_levels))


# ======================================================================
# Match runner — one pairing, N rounds
# ======================================================================
def run_match(env, agent1, agent2, n_rounds):
    """
    Run one pairing for exactly n_rounds steps.
    The env's max_steps is set to 1 in __main__ so each step() call
    truncates immediately — we then reset the env's internal counter
    while keeping each agent's history alive across rounds.
    This separates the env's "episode" concept from the tournament's
    "round count" concept, which is an important distinction.
    """
    agent1.reset()
    agent2.reset()
    env.reset()

    history1 = []   # agent1's view: [{"my_price", "opp_price"}, ...]
    history2 = []   # agent2's view

    log = []
    for round_num in range(n_rounds):
        a1 = agent1.act(history1)
        a2 = agent2.act(history2)

        obs, reward, terminated, truncated, info = env.step([a1, a2])
        p1, p2 = info["prices"]
        profit1, profit2 = info["profits"]

        history1.append({"my_price": float(p1), "opp_price": float(p2)})
        history2.append({"my_price": float(p2), "opp_price": float(p1)})

        log.append({
            "round":   round_num,
            "agent1":  agent1.name,
            "agent2":  agent2.name,
            "p1":      float(p1),
            "p2":      float(p2),
            "profit1": float(profit1),
            "profit2": float(profit2),
        })

        if truncated:
            # reset env step counter only — agents keep their histories
            env.reset()

    return log


# ======================================================================
# Tournament runner — round-robin, all pairings incl. self-play
# ======================================================================
def run_tournament(env, agents, n_rounds=1000, save_csv=True):
    """
    Round-robin: every unique pairing runs once, INCLUDING self-play
    (e.g. TFT vs TFT). Self-play reveals steady-state behaviour when
    both firms use the same strategy — essential for the report.

    Skips i > j to avoid running the same pair twice (the environment
    is symmetric so Nash-vs-TFT == TFT-vs-Nash in aggregate stats),
    but keeps i == j (self-play).
    """
    all_logs = {}
    summary_rows = []

    for i, a1 in enumerate(agents):
        for j, a2 in enumerate(agents):
            if i > j:
                continue    # skip true duplicate (swapped order, same pair)
            # i == j → self-play: keep it
            # i <  j → unique pairing: run it

            key = f"{a1.name}_vs_{a2.name}"
            log = run_match(env, a1, a2, n_rounds)
            all_logs[key] = log

            avg_p1      = float(np.mean([r["p1"]      for r in log]))
            avg_p2      = float(np.mean([r["p2"]      for r in log]))
            avg_profit1 = float(np.mean([r["profit1"] for r in log]))
            avg_profit2 = float(np.mean([r["profit2"] for r in log]))

            print(f"{key:45s}  "
                  f"avg_p1={avg_p1:6.2f}  avg_p2={avg_p2:6.2f}  "
                  f"avg_profit1={avg_profit1:8.2f}  avg_profit2={avg_profit2:8.2f}")

            summary_rows.append({
                "pairing":      key,
                "agent1":       a1.name,
                "agent2":       a2.name,
                "n_rounds":     n_rounds,
                "avg_p1":       round(avg_p1, 4),
                "avg_p2":       round(avg_p2, 4),
                "avg_profit1":  round(avg_profit1, 4),
                "avg_profit2":  round(avg_profit2, 4),
            })

    if save_csv:
        csv_path = "tournament_results.csv"
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=summary_rows[0].keys())
            writer.writeheader()
            writer.writerows(summary_rows)
        print(f"\nSummary saved → {csv_path}")

    return all_logs


# ======================================================================
# Main
# ======================================================================
if __name__ == "__main__":
    a, b, c = 100.0, 1.0, 10.0
    monopoly_price = (a + b * c) / (2 * b)   # = 55.0
    print(f"Monopoly price  = {monopoly_price}") #The monopoly price isn't saying "only one firm exists." It's saying: "if these two firms somehow cooperate perfectly and act as if they were one firm, what price would they charge?"
    print(f"Nash price (=c) = {c}\n")

    # max_steps=1: each env.step() call truncates after 1 step so that
    # run_match() can manage the round loop itself cleanly
    env = BertrandPricingEnv(a=a, b=b, marginal_cost=c,
                              n_price_levels=50, max_price=100, max_steps=1)
    grid = env.price_grid

    agents = [
        AlwaysNashAgent   ("AlwaysNash",    grid, marginal_cost=c),
        AlwaysColludeAgent("AlwaysCollude", grid, monopoly_price=monopoly_price),
        TitForTatAgent    ("TitForTat",     grid, monopoly_price=monopoly_price),
        RandomAgent       ("Random",        grid, seed=42),
    ]

    print("=== Round-robin tournament (1000 rounds per pairing) ===\n")
    logs = run_tournament(env, agents, n_rounds=1000)

Monopoly price  = 55.0
Nash price (=c) = 10.0

=== Round-robin tournament (1000 rounds per pairing) ===

AlwaysNash_vs_AlwaysNash                       avg_p1= 10.00  avg_p2= 10.00  avg_profit1=    0.00  avg_profit2=    0.00
AlwaysNash_vs_AlwaysCollude                    avg_p1= 10.00  avg_p2= 55.00  avg_profit1=    0.00  avg_profit2=    0.00
AlwaysNash_vs_TitForTat                        avg_p1= 10.00  avg_p2= 10.04  avg_profit1=    0.00  avg_profit2=    0.00
AlwaysNash_vs_Random                           avg_p1= 10.00  avg_p2= 30.13  avg_profit1=    0.00  avg_profit2= -248.32
AlwaysCollude_vs_AlwaysCollude                 avg_p1= 55.00  avg_p2= 55.00  avg_profit1= 1012.50  avg_profit2= 1012.50
AlwaysCollude_vs_TitForTat                     avg_p1= 55.00  avg_p2= 55.00  avg_profit1= 1012.50  avg_profit2= 1012.50
AlwaysCollude_vs_Random                        avg_p1= 55.00  avg_p2= 29.71  avg_profit1=  512.33  avg_profit2=   40.49
TitForTat_vs_TitForTat                         avg_p1= 

In [10]:
import numpy as np
import csv


# ======================================================================
# Monopoly profit — used for reward scaling
# pi_max = (a - b*c)^2 / (4*b)
# ======================================================================
def monopoly_profit(a, b, c):
    return (a - b * c) ** 2 / (4 * b)


# ======================================================================
# Q-Learning Agent
# ======================================================================
class QLearningAgent:
    """
    Tabular Q-learning with ε-greedy exploration.

    State:  (own_price_idx, opp_price_idx) — last round's two prices
            as indices into the price grid. This gives a fully
            observable, discrete Markov state.
    Action: price index in [0, n_price_levels - 1]
    Reward: scaled profit in [0, 1]

    Q-table shape: (n_price_levels, n_price_levels, n_price_levels)
                    own_last_price  opp_last_price  action
    """

    def __init__(
        self,
        name,
        n_price_levels,
        alpha=0.1,          # learning rate
        gamma=0.95,         # discount factor
        epsilon_start=1.0,  # start fully random (explore everything)
        epsilon_end=0.05,   # floor — never stop exploring completely
        epsilon_decay_frac=0.80,  # decay over this fraction of total episodes
        n_episodes=20_000,  # must match training loop
        reward_scale=1.0,   # divide raw profit by this (set to pi_max)
    ):
        self.name = name
        self.n = n_price_levels
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.reward_scale = reward_scale

        # Q-table: all zeros initially
        # Shape: [own_last_price_idx, opp_last_price_idx, action_idx]
        self.Q = np.zeros((n_price_levels, n_price_levels, n_price_levels),
                          dtype=np.float32)

        # Precompute epsilon decay per episode
        decay_episodes = int(n_episodes * epsilon_decay_frac)
        self._eps_decay_per_ep = (epsilon_start - epsilon_end) / max(decay_episodes, 1)

        # For logging
        self.episode_profits = []

    # ------------------------------------------------------------------
    def act(self, own_idx, opp_idx, greedy=False):
        """
        ε-greedy action selection.
        greedy=True forces exploitation (used during evaluation).
        """
        if not greedy and np.random.random() < self.epsilon:
            return np.random.randint(0, self.n)     # explore
        return int(np.argmax(self.Q[own_idx, opp_idx]))    # exploit

    # ------------------------------------------------------------------
    def update(self, own_idx, opp_idx, action, reward, next_own_idx, next_opp_idx):
        """
        Bellman update — the core of Q-learning:

        Q(s, a) ← Q(s, a) + α * [r + γ * max_a' Q(s', a') - Q(s, a)]

        where:
            s  = (own_idx, opp_idx)         current state
            a  = action                      price chosen
            r  = reward                      scaled profit
            s' = (next_own_idx, next_opp_idx) next state
        """
        current_q = self.Q[own_idx, opp_idx, action]

        # Best Q-value achievable from the next state
        best_next_q = np.max(self.Q[next_own_idx, next_opp_idx])

        # TD target
        td_target = reward + self.gamma * best_next_q

        # TD error
        td_error = td_target - current_q

        # Update
        self.Q[own_idx, opp_idx, action] += self.alpha * td_error

    # ------------------------------------------------------------------
    def decay_epsilon(self):
        """Call once per episode after training that episode."""
        self.epsilon = max(self.epsilon_end,
                           self.epsilon - self._eps_decay_per_ep)

    # ------------------------------------------------------------------
    def price_idx_from_obs(self, normalized_price, n_levels):
        """
        Convert a normalized price (0–1 float from env observation)
        back to a grid index.
        """
        return int(round(normalized_price * (n_levels - 1)))


# ======================================================================
# Training loop — two agents, self-play
# ======================================================================
def train(
    env,
    agent1,
    agent2,
    n_episodes=20_000,
    rounds_per_episode=50,
    log_every=1000,
):
    """
    Both agents train simultaneously.
    Each episode: reset env, run rounds_per_episode steps,
    both agents update their own Q-tables after every step.
    """
    n = env.n_price_levels
    pi_max = agent1.reward_scale   # same for both agents

    # Track average price per log window for convergence monitoring
    price_log = []

    for episode in range(n_episodes):
        obs, _ = env.reset()

        # Initial state: both prices start at 0 → index 0
        own1_idx = agent1.price_idx_from_obs(float(obs[0]), n)
        opp1_idx = agent1.price_idx_from_obs(float(obs[1]), n)
        own2_idx = agent2.price_idx_from_obs(float(obs[1]), n)
        opp2_idx = agent2.price_idx_from_obs(float(obs[0]), n)

        ep_profit1 = 0.0
        ep_profit2 = 0.0
        ep_prices1 = []

        for step in range(rounds_per_episode):
            # Both agents pick actions
            a1 = agent1.act(own1_idx, opp1_idx)
            a2 = agent2.act(own2_idx, opp2_idx)

            # Step environment
            next_obs, rewards, terminated, truncated, info = env.step([a1, a2])

            r1 = float(rewards[0]) / pi_max    # scale to [0,1]
            r2 = float(rewards[1]) / pi_max

            # Next state indices
            next_own1 = agent1.price_idx_from_obs(float(next_obs[0]), n)
            next_opp1 = agent1.price_idx_from_obs(float(next_obs[1]), n)
            next_own2 = agent2.price_idx_from_obs(float(next_obs[1]), n)
            next_opp2 = agent2.price_idx_from_obs(float(next_obs[0]), n)

            # Both agents update their Q-tables independently
            agent1.update(own1_idx, opp1_idx, a1, r1, next_own1, next_opp1)
            agent2.update(own2_idx, opp2_idx, a2, r2, next_own2, next_opp2)

            # Advance state
            own1_idx, opp1_idx = next_own1, next_opp1
            own2_idx, opp2_idx = next_own2, next_opp2

            ep_profit1 += float(rewards[0])
            ep_profit2 += float(rewards[1])
            ep_prices1.append(float(info["prices"][0]))

            if truncated:
                obs, _ = env.reset()
                own1_idx = agent1.price_idx_from_obs(float(obs[0]), n)
                opp1_idx = agent1.price_idx_from_obs(float(obs[1]), n)
                own2_idx = agent2.price_idx_from_obs(float(obs[1]), n)
                opp2_idx = agent2.price_idx_from_obs(float(obs[0]), n)

        agent1.episode_profits.append(ep_profit1)
        agent2.episode_profits.append(ep_profit2)

        # Decay epsilon after each episode (not each step)
        agent1.decay_epsilon()
        agent2.decay_epsilon()

        price_log.append(np.mean(ep_prices1))

        # Log progress
        if (episode + 1) % log_every == 0:
            window = price_log[-log_every:]
            avg_price = np.mean(window)
            print(f"Episode {episode+1:6d}/{n_episodes}  "
                  f"avg_price(agent1)={avg_price:6.2f}  "
                  f"ε={agent1.epsilon:.3f}")

    return price_log


# ======================================================================
# Evaluation — run greedy agents (no exploration) for N rounds
# ======================================================================
def evaluate(env, agent1, agent2, n_rounds=1000):
    """
    Run both agents greedily (ε=0) and report average prices + profits.
    This is the mid-project review check:
        Q-agent must beat the Random baseline average profit (~-48)
        and ideally converge toward the monopoly price zone (55).
    """
    obs, _ = env.reset()
    n = env.n_price_levels

    own1 = agent1.price_idx_from_obs(float(obs[0]), n)
    opp1 = agent1.price_idx_from_obs(float(obs[1]), n)
    own2 = agent2.price_idx_from_obs(float(obs[1]), n)
    opp2 = agent2.price_idx_from_obs(float(obs[0]), n)

    prices1, prices2, profits1, profits2 = [], [], [], []

    for _ in range(n_rounds):
        a1 = agent1.act(own1, opp1, greedy=True)
        a2 = agent2.act(own2, opp2, greedy=True)

        next_obs, rewards, _, truncated, info = env.step([a1, a2])

        prices1.append(float(info["prices"][0]))
        prices2.append(float(info["prices"][1]))
        profits1.append(float(rewards[0]))
        profits2.append(float(rewards[1]))

        own1 = agent1.price_idx_from_obs(float(next_obs[0]), n)
        opp1 = agent1.price_idx_from_obs(float(next_obs[1]), n)
        own2 = agent2.price_idx_from_obs(float(next_obs[1]), n)
        opp2 = agent2.price_idx_from_obs(float(next_obs[0]), n)

        if truncated:
            obs, _ = env.reset()
            own1 = agent1.price_idx_from_obs(float(obs[0]), n)
            opp1 = agent1.price_idx_from_obs(float(obs[1]), n)
            own2 = agent2.price_idx_from_obs(float(obs[1]), n)
            opp2 = agent2.price_idx_from_obs(float(obs[0]), n)

    print("\n=== Evaluation (greedy, 1000 rounds) ===")
    print(f"Agent1 avg price:  {np.mean(prices1):.2f}  "
          f"(Nash=10.0, Monopoly=55.0)")
    print(f"Agent2 avg price:  {np.mean(prices2):.2f}")
    print(f"Agent1 avg profit: {np.mean(profits1):.2f}  "
          f"(Random baseline ~ -48)")
    print(f"Agent2 avg profit: {np.mean(profits2):.2f}")

    collusion_idx = (np.mean(profits1) + np.mean(profits2)) / 2
    print(f"\nAvg joint profit per firm: {collusion_idx:.2f}")
    print(f"Monopoly per-firm profit (ceiling): 1012.50")
    print(f"Collusion index (% of monopoly): "
          f"{100 * collusion_idx / 1012.5:.1f}%")

    return {
        "avg_price1": np.mean(prices1),
        "avg_price2": np.mean(prices2),
        "avg_profit1": np.mean(profits1),
        "avg_profit2": np.mean(profits2),
    }


# ======================================================================
# Save training log to CSV
# ======================================================================
def save_training_log(price_log, path="q_learning_training_log.csv"):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["episode", "avg_price_agent1"])
        for i, p in enumerate(price_log):
            writer.writerow([i + 1, round(p, 4)])
    print(f"Training log saved → {path}")


# ======================================================================
# Main
# ======================================================================
if __name__ == "__main__":
    # --- environment ---
    a, b, c = 100.0, 1.0, 10.0
    N_LEVELS   = 50
    N_EPISODES = 20_000
    ROUNDS_PER_EP = 50

    env = BertrandPricingEnv(
        a=a, b=b, marginal_cost=c,
        n_price_levels=N_LEVELS,
        max_price=100.0,
        max_steps=ROUNDS_PER_EP,
    )

    pi_max = monopoly_profit(a, b, c)   # = 2025.0 (reward scaling denominator)
    print(f"Max possible profit (monopoly, single firm): {pi_max}")
    print(f"Price grid: {N_LEVELS} levels from 0 to 100")
    print(f"Q-table size per agent: {N_LEVELS}x{N_LEVELS}x{N_LEVELS} = "
          f"{N_LEVELS**3:,} entries\n")

    np.random.seed(42)   # reproducibility

    # --- agents ---
    agent1 = QLearningAgent(
        name="QLearner_1",
        n_price_levels=N_LEVELS,
        alpha=0.1,
        gamma=0.95,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay_frac=0.80,   # decay over 80% of episodes
        n_episodes=N_EPISODES,
        reward_scale=pi_max,       # scale profits to [0,1]
    )
    agent2 = QLearningAgent(
        name="QLearner_2",
        n_price_levels=N_LEVELS,
        alpha=0.1,
        gamma=0.95,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay_frac=0.80,
        n_episodes=N_EPISODES,
        reward_scale=pi_max,
    )

    # --- train ---
    print("=== Training (self-play, 20,000 episodes × 50 rounds) ===")
    price_log = train(
        env, agent1, agent2,
        n_episodes=N_EPISODES,
        rounds_per_episode=ROUNDS_PER_EP,
        log_every=2000,
    )

    # --- evaluate (mid-project review gate) ---
    results = evaluate(env, agent1, agent2, n_rounds=1000)

    # --- save logs ---
    save_training_log(price_log)

    # --- mid-project review check ---
    print("\n=== Mid-Project Review Gate ===")
    random_baseline_profit = -48.0   # from tournament results
    if results["avg_profit1"] > random_baseline_profit:
        print("PASS: Q-agent beats Random baseline")
    else:
        print("FAIL: Q-agent does not beat Random. Check environment or "
              "reward function for bugs before proceeding.")

Max possible profit (monopoly, single firm): 2025.0
Price grid: 50 levels from 0 to 100
Q-table size per agent: 50x50x50 = 125,000 entries

=== Training (self-play, 20,000 episodes × 50 rounds) ===
Episode   2000/20000  avg_price(agent1)= 30.88  ε=0.881
Episode   4000/20000  avg_price(agent1)= 32.31  ε=0.763
Episode   6000/20000  avg_price(agent1)= 33.86  ε=0.644
Episode   8000/20000  avg_price(agent1)= 35.47  ε=0.525
Episode  10000/20000  avg_price(agent1)= 37.15  ε=0.406
Episode  12000/20000  avg_price(agent1)= 38.81  ε=0.288
Episode  14000/20000  avg_price(agent1)= 40.47  ε=0.169
Episode  16000/20000  avg_price(agent1)= 41.62  ε=0.050
Episode  18000/20000  avg_price(agent1)= 41.53  ε=0.050
Episode  20000/20000  avg_price(agent1)= 42.28  ε=0.050

=== Evaluation (greedy, 1000 rounds) ===
Agent1 avg price:  41.42  (Nash=10.0, Monopoly=55.0)
Agent2 avg price:  43.08
Agent1 avg profit: 808.31  (Random baseline ~ -48)
Agent2 avg profit: 789.47

Avg joint profit per firm: 798.89
Monopoly p